# U.S. Inflation State-Space Modeling

**Author: QIDIAN XING**

## Maximum-Likelihood Estimation and Parameter Recovery

**Research objective:** simulate and estimate T=300, 3000 and 300000; report truth, estimates, SEs and convergence. The existing repeated Monte Carlo appears in a supplementary appendix.

This notebook is independent: start its kernel in notebooks/ and use Restart Kernel / Run All. No other notebook or project module is required. AI assistance is summarized in [AI Assistance and Validation](../docs/AI_Assistance_and_Validation.md).


## Parameters and fixed seeds

Monthly rates are in percentage points. Q and H are variances. Experimental settings are unchanged from the verified project.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import hashlib
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from numba import njit
from scipy.optimize import minimize
from IPython.display import display, HTML

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2, 'figure.dpi': 120,
})
COLORS = {'monthly': '#1e4e79', 'annual': '#9b4b2b',
          'benchmark': '#52665f', 'data': '#454545'}

In [2]:
CONFIG = {'true_parameters': {'mu': 0.2, 'phi': 0.9, 'Q': 0.04, 'H': 0.01},
 'sample_lengths': [300, 3000, 300000],
 'simulation_seeds': [431300, 4313000, 431300000],
 'economic_simulation_seeds': [43101, 43102, 43103, 43104],
 'economic_sample_length': 936,
 'optimizer': {'initial_phi': [-0.5, 0.3, 0.8, 0.95],
               'maxiter': 1200,
               'gtol': 1e-08,
               'gradient_step': 1e-05},
 'hessian_steps': [0.002, 0.001, 0.0005],
 'input_csv': '../data/US_inflation47_25.csv',
 'monte_carlo': {'sample_length': 936,
                 'replications': 200,
                 'master_seed': 20261003,
                 'gradient_tolerance': 1e-06,
                 'parameter_valid_definition': 'Finite interior estimate, finite full '
                                               'likelihood, selected actual average '
                                               'gradient infinity norm < 1e-6. Optimizer '
                                               'flag is counted separately.',
                 'se_valid_definition': 'Parameter-valid plus returned fit; positive '
                                        'full-likelihood Hessian; finite positive SEs; '
                                        'original gradient and <2% step-stability criteria '
                                        'pass for all four parameters.',
                 'denominator_rule': 'All primary calibration columns use the same '
                                     'all-parameter SE-valid complete-case set; companion '
                                     'columns describe all parameter-valid estimates.',
                 'provenance': 'Predeclared supplementary uncertainty-calibration study; '
                               'independent streams and all attempts are retained.'}}

### Innovation likelihood and model-based asymptotic SEs

Before observing $y_t$, form $a_{t|t-1}$ and $P_{t|t-1}$. After receiving
$y_t$, compute its innovation and likelihood **before** updating the state:
$$v_t=y_t-\mu-a_{t|t-1},\quad F_t=P_{t|t-1}+H,\quad K_t=P_{t|t-1}/F_t.$$
$$a_{t|t}=a_{t|t-1}+K_tv_t,\quad P_{t|t}=P_{t|t-1}H/F_t.$$
$$a_{t+1|t}=\phi a_{t|t},\quad P_{t+1|t}=\phi^2P_{t|t}+Q,$$
$$\ell(\theta)=-\tfrac12\sum_t[\log(2\pi)+\log F_t+v_t^2/F_t].$$

BFGS minimizes the average negative likelihood from four data-derived starts.
The unconstrained coordinates are $u=(\mu,b,\log Q,\log H)$ with
$\phi=\tanh b$. SEs use the Hessian $I_u=\nabla_u^2[-\ell(u)]$ of the
**actual full likelihood**, including derivatives of its stationary prior.
No optimizer inverse-Hessian approximation or pseudoinverse supplies inference.
$$g=(\mu,\phi,\sqrt Q,\sqrt H),\quad
J=\operatorname{diag}(1,1-\phi^2,\sqrt Q/2,\sqrt H/2),$$
$$\operatorname{Cov}(\widehat g)=JI_u^{-1}J',\qquad
\operatorname{SE}(\widehat g_j)=\sqrt{[JI_u^{-1}J']_{jj}}.$$
An average-likelihood Hessian would require multiplication by the observation
count. Three central-difference scales check positive curvature and SE stability;
the middle scale supplies the reported covariance. These are model-based
**asymptotic Hessian/delta-method SEs**, not finite-sample exact or robust SEs.

## Simulation and accelerated likelihood

The Numba recursion is identical to the verified implementation. File caching is disabled because a notebook has no stable source-module cache location; in-kernel JIT acceleration remains active.

In [3]:
@dataclass(frozen=True)
class Parameters:
    mu: float
    phi: float
    Q: float
    H: float

    def __post_init__(self):
        if not (abs(self.phi) < 1 and self.Q > 0 and self.H > 0):
            raise ValueError("Require |phi|<1, Q>0 and H>0")

    def vector(self):
        return np.array([self.mu, self.phi, self.Q, self.H])

    def reported(self):
        return np.array([self.mu, self.phi, np.sqrt(self.Q), np.sqrt(self.H)])

In [4]:
def simulate(T_sample, p, seed):
    """Use a stationary initial draw and independent state/observation shocks."""
    rng = np.random.default_rng(seed)
    alpha = np.empty(T_sample)
    alpha[0] = rng.normal(0, np.sqrt(p.Q / (1 - p.phi**2)))
    epsilon = rng.normal(0, np.sqrt(p.Q), T_sample - 1)
    eta = rng.normal(0, np.sqrt(p.H), T_sample)
    for t in range(1, T_sample):
        alpha[t] = p.phi * alpha[t-1] + epsilon[t-1]
    return p.mu + alpha + eta, alpha, eta

In [5]:
def from_u(u):
    return Parameters(float(u[0]), float(np.tanh(u[1])),
                      float(np.exp(u[2])), float(np.exp(u[3])))

In [6]:
def to_u(p):
    return np.array([p.mu, np.arctanh(p.phi), np.log(p.Q), np.log(p.H)])

In [7]:
@njit(cache=False)
def monthly_nll_fast(y, mu, phi, Q, H):
    a = 0.0
    P = Q / (1 - phi*phi)
    nll = 0.0
    for yt in y:
        v = yt - mu - a
        F = P + H
        if F <= 0 or not np.isfinite(F):
            return np.inf
        nll += 0.5 * (np.log(2*np.pi) + np.log(F) + v*v/F)
        K = P / F
        a = phi * (a + K*v)
        P = phi*phi * (P*H/F) + Q
    return nll

In [8]:
def gradient(f, x, step=1e-5):
    h = step*np.maximum(1, np.abs(x))
    g = np.empty(len(x))
    for i in range(len(x)):
        e = np.zeros(len(x)); e[i] = h[i]
        g[i] = (f(x+e)-f(x-e))/(2*h[i])
    return g

In [9]:
def hessian(f, x, step):
    h = step*np.maximum(1, np.abs(x))
    M = np.empty((len(x),len(x)))
    f0 = f(x)
    for i in range(len(x)):
        ei = np.zeros(len(x)); ei[i] = h[i]
        M[i,i] = (f(x+ei)-2*f0+f(x-ei))/h[i]**2
        for j in range(i):
            ej = np.zeros(len(x)); ej[j] = h[j]
            M[i,j] = M[j,i] = (f(x+ei+ej)-f(x+ei-ej)-f(x-ei+ej)+f(x-ei-ej))/(4*h[i]*h[j])
    return M

In [10]:
def fit(observations, kind, config, diagnostics=None):
    """Fit the unchanged MLE; optional observer retains diagnostics on failure."""
    data = np.ascontiguousarray(observations, dtype=float)
    if kind != 'monthly':
        raise ValueError('This notebook estimates the monthly model.')
    core = monthly_nll_fast
    def full_nll(u):
        # Numerical overflow guards leave the economically relevant region broad.
        if not np.isfinite(u).all() or abs(np.tanh(u[1])) >= 1 or np.any(np.abs(u[2:])>40):
            return 1e100
        p = from_u(u)
        return float(core(data, *p.vector()))
    objective = lambda u: full_nll(u)/len(data)
    cfg = config['optimizer']
    jac = lambda u: gradient(objective, u, cfg['gradient_step'])
    sample_mean = float(data.mean()/(12 if kind == 'annual' else 1))
    # Start variances are data-derived, never copied from the simulation truth.
    variance = float(np.var(data)/(80 if kind == 'annual' else 1))
    starts = []
    records = []
    for phi in cfg['initial_phi']:
        p0 = Parameters(sample_mean, phi, max(variance*0.6*(1-phi**2),1e-4),max(variance*0.4,1e-4))
        u0 = to_u(p0)
        start = time.perf_counter()
        result = minimize(objective,u0,jac=jac,method='BFGS',
                          options={'gtol':cfg['gtol'],'maxiter':cfg['maxiter']})
        records.append(dict(method='BFGS',initial_phi=phi,initial_u=u0.tolist(),
                            success=bool(result.success),message=str(result.message),
                            iterations=int(result.nit),nfev=int(result.nfev),
                            nll=full_nll(result.x),u=result.x.tolist(),
                            gradient_inf=float(np.max(np.abs(jac(result.x)))),
                            seconds=time.perf_counter()-start))
        starts.append(result)
    result = min(starts,key=lambda r:r.fun)
    u = result.x
    p = from_u(u)
    nll = full_nll(u)
    if diagnostics is not None:
        diagnostics(dict(stage='optimization', parameters=dict(mu=p.mu,phi=p.phi,Q=p.Q,H=p.H),
                         reported=p.reported().tolist(), loglik=-nll,
                         optimizer_success=bool(result.success), optimizer_message=str(result.message),
                         gradient_average_inf=float(np.max(np.abs(jac(u)))),
                         optimization_runs=records))
    hs = [hessian(full_nll,u,s) for s in config['hessian_steps']]
    I = hs[1]
    eigenvalues = np.linalg.eigvalsh(I)
    if diagnostics is not None:
        diagnostics(dict(stage='curvature',hessian=I.tolist(),eigenvalues=eigenvalues.tolist(),
                         condition_number=float(np.linalg.cond(I))))
    if eigenvalues.min() <= 0:
        raise RuntimeError(f'{kind}: actual likelihood Hessian is not positive definite: {eigenvalues}')
    covariance_u = np.linalg.solve(I,np.eye(4))
    J = np.diag([1,1-p.phi**2,np.sqrt(p.Q)/2,np.sqrt(p.H)/2])
    covariance_g = J@covariance_u@J.T
    se = np.sqrt(np.diag(covariance_g))
    Jv = np.diag([1,1-p.phi**2,p.Q,p.H])
    se_variance = np.sqrt(np.diag(Jv@covariance_u@Jv.T))
    variants = [np.sqrt(np.diag(J@np.linalg.solve(h,np.eye(4))@J.T)) for h in hs]
    se_relative_change = float(max(np.max(np.abs(v/se-1)) for v in variants))
    grad = gradient(full_nll,u,config['optimizer']['gradient_step'])
    diag = dict(hessian_steps=config['hessian_steps'],hessian=I.tolist(),
                eigenvalues=eigenvalues.tolist(),condition_number=float(np.linalg.cond(I)),
                symmetry_error=float(np.max(np.abs(I-I.T))),
                hessian_relative_changes=[float(np.linalg.norm(h-I)/np.linalg.norm(I)) for h in hs],
                se_relative_change=se_relative_change,gradient_full=grad.tolist(),
                gradient_average_inf=float(np.max(np.abs(grad))/len(data)),
                covariance_u=covariance_u.tolist(),covariance_reported=covariance_g.tolist(),
                reliable=bool(se_relative_change<0.02 and np.max(np.abs(grad))/len(data)<1e-6))
    if diagnostics is not None:
        diagnostics(dict(stage='inference',se=se.tolist(),inference=diag))
    if not diag['reliable']:
        raise RuntimeError(f'{kind}: unreliable inference diagnostics: {diag}')
    return dict(kind=kind,n=len(data),parameters=dict(mu=p.mu,phi=p.phi,Q=p.Q,H=p.H),
                sqrt_Q=np.sqrt(p.Q),sqrt_H=np.sqrt(p.H),se=se.tolist(),
                se_variances=se_variance.tolist(),u=u.tolist(),loglik=-nll,
                phi12=p.phi**12,optimizer_success=bool(result.success),
                optimizer_message=str(result.message),optimization_runs=records,
                inference=diag)

In [11]:
def parameter_table(result, truth=None):
    p = Parameters(**result['parameters'])
    table = pd.DataFrame({'Parameter': ['mu', 'phi', 'sqrt_Q', 'sqrt_H'],
                          'Estimate': p.reported(), 'SE': result['se']})
    if truth is not None:
        table.insert(1, 'True', truth.reported())
    return table


def convergence_table(results):
    return pd.DataFrame([
        {'Fit': name, 'Observations': r['n'], 'Log likelihood': r['loglik'],
         'Optimizer success': r['optimizer_success'],
         'Message': r['optimizer_message'],
         'Average gradient': r['inference']['gradient_average_inf'],
         'Minimum Hessian eigenvalue': min(r['inference']['eigenvalues']),
         'Relative SE step variation': r['inference']['se_relative_change']}
        for name, r in results.items()
    ])


def show_table(frame, digits=6):
    display(HTML(frame.to_html(index=False, float_format=lambda value: f'{value:.{digits}f}')))


def calendar_axis(axis):
    axis.xaxis.set_major_locator(mdates.YearLocator(20))
    axis.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

## Parameter recovery across three sample lengths

In [12]:
truth = Parameters(**CONFIG['true_parameters'])
simulation_results = {}
rows = []
for n, seed in zip(CONFIG['sample_lengths'], CONFIG['simulation_seeds']):
    sample, _, _ = simulate(n, truth, seed)
    result = fit(sample, 'monthly', CONFIG)
    simulation_results[f'simulation_{n}'] = result
    for name, true, estimate, se in zip(['mu','phi','sqrt_Q','sqrt_H'], truth.reported(),
                                       Parameters(**result['parameters']).reported(), result['se']):
        rows.append({'T_sample': n, 'Seed': seed, 'Parameter': name,
                     'True': true, 'Estimate': estimate, 'SE': se})
simulation_table = pd.DataFrame(rows)
show_table(simulation_table)
show_table(convergence_table(simulation_results))
assert monthly_nll_fast.signatures, 'Numba acceleration was not compiled.'

T_sample,Seed,Parameter,True,Estimate,SE
300,431300,mu,0.200000,0.194771,0.083104
300,431300,phi,0.900000,0.851541,0.036187
300,431300,sqrt_Q,0.200000,0.217407,0.019021
300,431300,sqrt_H,0.100000,0.066681,0.033588
3000,4313000,mu,0.200000,0.188312,0.038830
3000,4313000,phi,0.900000,0.906462,0.008810
3000,4313000,sqrt_Q,0.200000,0.199325,0.005567
3000,4313000,sqrt_H,0.100000,0.104425,0.006178
300000,431300000,mu,0.200000,0.197610,0.003627
300000,431300000,phi,0.900000,0.898970,0.000926


Fit,Observations,Log likelihood,Optimizer success,Message,Average gradient,Minimum Hessian eigenvalue,Relative SE step variation
simulation_300,300,9.685603,True,Optimization terminated successfully.,0.000000,0.959772,0.000020
simulation_3000,3000,15.834707,True,Optimization terminated successfully.,0.000000,60.652627,0.000004
simulation_300000,300000,4159.961852,True,Optimization terminated successfully.,0.000000,5229.059522,0.000004


Larger samples recover the truth more closely overall and sharply reduce uncertainty; an individual estimate need not improve monotonically. The 300-month observation-noise estimate is imprecise and remains in the table. Convergence flags are displayed alongside actual gradients, Hessian positivity and SE step stability.

## Supplementary appendix: T=936, B=200 uncertainty calibration

This supplementary repeated-sampling study evaluates uncertainty calibration
under the assumed model. The original truth, master seed
20261003 and all 200 independent SeedSequence streams are retained. Seeds are
generated before seeing any sample using the original spawn/generate_state method;
a fingerprint below checks their exact pre-saved specification.

Parameter-valid means finite interior parameters and likelihood with actual average
gradient below 1e-6. SE-valid additionally requires a returned fit, positive full
Hessian, finite positive SEs and the original gradient/less-than-2% SE-step checks
for every parameter. Optimizer flags are reported separately; a false flag alone
is not an exclusion. Every attempt stays visible, including errors and partial
diagnostics. All primary summaries use the same all-parameter SE-valid set;
companion summaries retain all parameter-valid estimates.

In [13]:
children = np.random.SeedSequence(CONFIG['monte_carlo']['master_seed']).spawn(CONFIG['monte_carlo']['replications'])
mc_streams = [dict(replication=i+1, entropy=int(child.entropy), spawn_key=list(child.spawn_key),
                   seed=int(child.generate_state(1, dtype=np.uint64)[0]))
              for i, child in enumerate(children)]
import json
stream_fingerprint = hashlib.sha256(json.dumps(mc_streams, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
assert stream_fingerprint == '1766386b7141eb4963e939ae64185b2a96981782f31b32a1ad160aef66382de7', 'The original MC streams changed.'
NAMES = ['mu', 'phi', 'sqrt_Q', 'sqrt_H']

In [14]:
def summarize(frame, truth):
    """Use the identical all-parameter SE-valid set for every calibration column."""
    valid = frame.loc[frame.se_valid.astype(bool)]
    parameter_valid = frame.loc[frame.parameter_valid.astype(bool)]
    rows = []
    for name, true in zip(NAMES, truth.reported()):
        estimates = valid[name].to_numpy(dtype=float)
        ses = valid['se_'+name].to_numpy(dtype=float)
        n = len(estimates)
        sd = float(np.std(estimates, ddof=1)) if n > 1 else np.nan
        mean = float(np.mean(estimates)) if n else np.nan
        mean_se = float(np.mean(ses)) if n else np.nan
        coverage = float(np.mean(np.abs(estimates-true) <= 1.959963984540054*ses)) if n else np.nan
        all_estimates = parameter_valid[name].to_numpy(dtype=float)
        rows.append(dict(parameter=name, true=true, mean_estimate=mean, bias=mean-true,
                         empirical_sd=sd, mean_se=mean_se, calibration_ratio=mean_se/sd if sd > 0 else np.nan,
                         n_matched=n, n_parameter_valid=len(all_estimates),
                         parameter_valid_mean=float(np.mean(all_estimates)) if len(all_estimates) else np.nan,
                         parameter_valid_bias=float(np.mean(all_estimates)-true) if len(all_estimates) else np.nan,
                         parameter_valid_sd=float(np.std(all_estimates,ddof=1)) if len(all_estimates)>1 else np.nan,
                         coverage_95=coverage, coverage_mcse=np.sqrt(coverage*(1-coverage)/n) if n else np.nan,
                         mean_bias_mcse=sd/np.sqrt(n) if n>1 else np.nan,
                         sd_mcse_normal_reference=sd/np.sqrt(2*(n-1)) if n>1 else np.nan))
    return pd.DataFrame(rows)

In [15]:
def run_monte_carlo(config, streams, truth):
    """Keep every attempted fit, including partial diagnostics and exceptions."""
    settings = config['monte_carlo']
    rows = []
    for stream in streams:
        timer = time.perf_counter()
        row = dict(replication=stream['replication'], seed=str(stream['seed']),
                   parameter_valid=False, se_valid=False, optimizer_success=None,
                   returned=False, error_type='', error_message='',
                   gradient_average_inf=np.nan, min_hessian_eigenvalue=np.nan,
                   hessian_condition=np.nan, se_step_relative_change=np.nan)
        row.update({name:np.nan for name in NAMES})
        row.update({'se_'+name:np.nan for name in NAMES})

        def observe(event):
            if event['stage'] == 'optimization':
                row.update(zip(NAMES,event['reported']))
                row.update(optimizer_success=event['optimizer_success'],
                           optimizer_message=event['optimizer_message'],
                           loglik=event['loglik'], gradient_average_inf=event['gradient_average_inf'])
                p = event['parameters']
                row['parameter_valid'] = bool(np.isfinite(event['reported']).all() and
                    abs(p['phi'])<1 and p['Q']>0 and p['H']>0 and np.isfinite(event['loglik']) and
                    event['gradient_average_inf']<settings['gradient_tolerance'])
            elif event['stage'] == 'curvature':
                row.update(min_hessian_eigenvalue=min(event['eigenvalues']),
                           hessian_condition=event['condition_number'])
            else:
                row.update({'se_'+name:value for name,value in zip(NAMES,event['se'])})
                row['se_step_relative_change'] = event['inference']['se_relative_change']

        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error', RuntimeWarning)
                y, _, _ = simulate(settings['sample_length'], truth, stream['seed'])
                row['sample_sha256'] = hashlib.sha256(y.astype('<f8').tobytes()).hexdigest()
                result = fit(y, 'monthly', config, diagnostics=observe)
                row['returned'] = True
                row['se_valid'] = bool(row['parameter_valid'] and result['inference']['reliable'] and
                    row['min_hessian_eigenvalue']>0 and np.isfinite(result['se']).all() and
                    np.all(np.asarray(result['se'])>0))
        except Exception as error:
            row['error_type'] = type(error).__name__
            row['error_message'] = str(error)
        row['seconds'] = time.perf_counter()-timer
        rows.append(row)
    return pd.DataFrame(rows)

In [16]:
mc_replications = run_monte_carlo(CONFIG, mc_streams, truth)
mc_calibration = summarize(mc_replications, truth)
flags = mc_replications.optimizer_success
mc_counts = pd.DataFrame([{
    'Attempted': len(mc_replications), 'Parameter-valid': int(mc_replications.parameter_valid.sum()),
    'SE-valid': int(mc_replications.se_valid.sum()), 'Exceptions': int((~mc_replications.returned).sum()),
    'Optimizer flag failures': int((flags == False).sum()), 'Optimizer flag unavailable': int(flags.isna().sum()),
}])
display(mc_counts)
show_table(mc_calibration[['parameter','true','mean_estimate','bias','empirical_sd','mean_se',
                           'calibration_ratio','coverage_95','n_matched','n_parameter_valid']])

,Attempted,Parameter-valid,SE-valid,Exceptions,Optimizer flag failures,Optimizer flag unavailable
0,200,200,200,0,0,0


parameter,true,mean_estimate,bias,empirical_sd,mean_se,calibration_ratio,coverage_95,n_matched,n_parameter_valid
mu,0.200000,0.183611,-0.016389,0.065672,0.062477,0.951364,0.915000,200,200
phi,0.900000,0.892803,-0.007197,0.018737,0.017120,0.913684,0.940000,200,200
sqrt_Q,0.200000,0.201300,0.001300,0.009511,0.010141,1.066188,0.940000,200,200
sqrt_H,0.100000,0.097042,-0.002958,0.012236,0.012306,1.005757,0.970000,200,200


### All replication outcomes

The scrollable table retains every attempt, original seed, sample hash, validity flag, estimate, SE, likelihood, convergence diagnostic and error field at full floating-point precision. It is generated by the computation above, not read from prior results.

In [17]:
display(HTML('<div style="max-height:520px;overflow:auto">' +
             mc_replications.to_html(index=False, float_format=lambda v: f'{v:.17g}') + '</div>'))
show_table(mc_calibration)

replication,seed,parameter_valid,se_valid,optimizer_success,returned,error_type,error_message,gradient_average_inf,min_hessian_eigenvalue,hessian_condition,se_step_relative_change,mu,phi,sqrt_Q,sqrt_H,se_mu,se_phi,se_sqrt_Q,se_sqrt_H,sample_sha256,optimizer_message,loglik,seconds
1,14193761781245560257,True,True,True,True,,,1.2424089534945892e-09,22.415600205180901,12.251792849068757,2.5908335468116661e-06,0.3216538696149342,0.90640398168873926,0.18283014340616549,0.10580659562561674,0.063318894944733597,0.016048123178458301,0.0095417103476358352,0.0099952861713936171,f970cf01aaa4189744ff5fefa5865e301ca198a9ef4657497dde8d09dada7d15,Optimization terminated successfully.,56.341479338499184,0.013358833006350324
2,8362603143905466707,True,True,True,True,,,7.7410359874296491e-10,5.6855204856781247,61.426686126786038,1.5051185753622853e-05,0.23889845323954714,0.86527850805382434,0.22177787294034029,0.080949566248813457,0.053513181518163658,0.019557168192725415,0.011022172537483747,0.016559670375946651,16b454340b8e5809f7996f8769cf17875adf6280244d6cd00836156cd8f42edc,Optimization terminated successfully.,-12.913970003661372,0.013149749996955507
3,6807466578896189815,True,True,True,True,,,3.1218933900845013e-09,13.481727842998012,22.3778367084739,5.8942787388716056e-06,0.10947596758557927,0.90259857877977379,0.20052199252049221,0.094214729649321927,0.066735705089323666,0.016191860581379006,0.010062874910720745,0.01209179711029652,9637a07dd53915c9c3333b3c59f0c71d975927ecb893bf6c5592ecb0d6b7ed31,Optimization terminated successfully.,27.257031119976759,0.013479207991622388
4,18344020339818064641,True,True,True,True,,,6.1521968075517464e-10,18.838564486394873,14.970344323219745,3.6009045057960876e-06,0.27324643752257444,0.90885771144432514,0.19635136001357487,0.105999157837806,0.069778758114726919,0.015634132468520769,0.010188278851759642,0.011146056935194961,4481945d7b940b75060ee3b74dae35acd4f367dfaaca8590d608ce1dc12b23b8,Optimization terminated successfully.,9.7996541684287841,0.013268249997054227
5,8270455949508450869,True,True,True,True,,,2.117397901387101e-10,2.2064796690104092,190.19930036884685,2.1856916112672486e-05,0.39644819067953602,0.85376258372480185,0.21957828086277514,0.058521764641960815,0.048816128782452582,0.019890386799696586,0.010011734995074215,0.019545502381410822,cca5e13b733df40b2b7b0545d46fe36afe4ced3a2b923de0d5a9e6c3f25317fa,Optimization terminated successfully.,37.44762910358196,0.013691250002011657
6,3103753594691751838,True,True,True,True,,,9.4258346020391674e-10,19.261616268122275,14.672933154850011,3.2086734851333176e-06,0.16491873041812741,0.90494835371966864,0.20111025015264042,0.1091864315730896,0.068573059742677425,0.015992485112716611,0.010397324689054058,0.01134627711013189,6dc5ac94ee3a98d71fc7e160277367a34c8d7dc1e814b10bcb55770d1c80a11f,Optimization terminated successfully.,-13.674378770959372,0.013310833004652523
7,3890402100683395008,True,True,True,True,,,1.5267075929533024e-09,11.29876880810853,27.962632325988768,8.3042113474540713e-06,0.19077920872177653,0.88749507972244268,0.2001467202535957,0.087391401229363563,0.057744494780299506,0.017432965740507688,0.0097576179889339084,0.012421022812346867,fbf3bc245c8ea7020333a5189519332ff555493ef01c4fad03d85ceb72f5d6c1,Optimization terminated successfully.,47.34128986415341,0.013187791992095299
8,12988190991760168501,True,True,True,True,,,9.433426262361877e-10,7.1059887594301205,47.701295466902266,1.3012378280219394e-05,0.2014017871080595,0.88148314569328612,0.20985370442906207,0.078259726291032713,0.057484678092041476,0.017745179847637436,0.0099306133253653731,0.014292423522924244,81219ab77c727e1affdc8aaad300d3270b4da59cbb7021463c457476d3b94c13,Optimization terminated successfully.,33.860218596347686,0.012850999992224388
9,14107057647617328261,True,True,True,True,,,4.6668398312466053e-10,13.170773102280116,23.48945294439611,6.761440038194344e-06,0.20714374381573439,0.88447694935836996,0.20659711053958787,0.095672106107293223,0.058075348291509087,0.017757252

parameter,true,mean_estimate,bias,empirical_sd,mean_se,calibration_ratio,n_matched,n_parameter_valid,parameter_valid_mean,parameter_valid_bias,parameter_valid_sd,coverage_95,coverage_mcse,mean_bias_mcse,sd_mcse_normal_reference
mu,0.200000,0.183611,-0.016389,0.065672,0.062477,0.951364,200,200,0.183611,-0.016389,0.065672,0.915000,0.019720,0.004644,0.003292
phi,0.900000,0.892803,-0.007197,0.018737,0.017120,0.913684,200,200,0.892803,-0.007197,0.018737,0.940000,0.016793,0.001325,0.000939
sqrt_Q,0.200000,0.201300,0.001300,0.009511,0.010141,1.066188,200,200,0.201300,0.001300,0.009511,0.940000,0.016793,0.000673,0.000477
sqrt_H,0.100000,0.097042,-0.002958,0.012236,0.012306,1.005757,200,200,0.097042,-0.002958,0.012236,0.970000,0.012062,0.000865,0.000613


In [18]:
ratios = ', '.join(f'{r.parameter}: {r.calibration_ratio:.3f}' for r in mc_calibration.itertuples())
display(HTML(f'<p>Mean SE / empirical SD: {ratios}. Coverage ranges from '
             f'{mc_calibration.coverage_95.min():.3f} to {mc_calibration.coverage_95.max():.3f}. '
             'A fixed batch has finite Monte Carlo uncertainty; neither ratio=1 nor coverage=0.95 is required. '
             'The batch mean alone does not identify unconditional estimator bias. No seed is replaced.</p>'))